# 01. Data Preprocessing

이 노트북은 RavenStack SaaS 고객 데이터를 불러와,  
모델 학습에 활용할 수 있는 형태로 전처리하는 과정을 담고 있다.

주요 작업은 다음과 같다.

- 원천 데이터 로드
- 날짜형 컬럼 변환
- feature usage 집계
- support ticket 집계
- account 기준 데이터 통합
- 파생 변수 생성
- 결측치 처리 및 플래그 생성
- 범주형 변수 인코딩
- 최종 학습용 데이터 저장

이 단계의 핵심은 서로 다른 테이블 grain을 **account 단위 분석 테이블**로 통합하는 데 있다.

## 1. 원천 데이터 로드

먼저 프로젝트에 사용되는 주요 원천 데이터를 불러온다.

사용하는 테이블은 다음과 같다.

- `accounts`: 고객 계정 기본 정보
- `subscriptions`: 구독 이력 정보
- `feature_usage`: 서비스 기능 사용 로그
- `support_tickets`: 고객 지원 티켓 정보

이 데이터들은 각각 서로 다른 단위를 가지므로,  
이후 전처리 단계에서 공통 분석 단위인 `account_id` 기준으로 정리하게 된다.

In [10]:
# preprocess.py
import pandas as pd
import numpy as np
# 1. 데이터 로드
accounts = pd.read_csv('../../data/raw/ravenstack_accounts.csv')
subscriptions = pd.read_csv('../../data/raw/ravenstack_subscriptions.csv')
usage = pd.read_csv('../../data/raw/ravenstack_feature_usage.csv')
tickets = pd.read_csv('../../data/raw/ravenstack_support_tickets.csv')

# [보강] 날짜 형식 변환
accounts['signup_date'] = pd.to_datetime(accounts['signup_date'])
usage['usage_date'] = pd.to_datetime(usage['usage_date'])

# 2. Feature Usage 보강 집계 (계정별 사용 패턴 분석)
usage_with_acc = pd.merge(usage, subscriptions[['subscription_id', 'account_id']], on='subscription_id', how='left')
usage_summary = usage_with_acc.groupby('account_id').agg({
    'usage_count': 'sum',
    'usage_duration_secs': 'sum',
    'error_count': 'sum',
    'feature_name': 'nunique',  # 얼마나 다양한 기능을 썼는지 (Feature Diversity)
    'is_beta_feature': 'max'     # 베타 기능을 한 번이라도 써봤는지 (1/0)
}).reset_index()
usage_summary.columns = ['account_id', 'total_usage_cnt', 'total_duration', 'total_errors', 'feature_diversity', 'used_beta']

# 3. Support Tickets 보강 (결측 플래그 추가)
ticket_summary = tickets.groupby('account_id').agg({
    'ticket_id': 'count',
    'satisfaction_score': 'mean',
    'resolution_time_hours': 'mean'
}).reset_index()
ticket_summary.columns = ['account_id', 'total_tickets', 'avg_satisfaction', 'avg_resolution_time']

# 4. 전체 데이터 통합
df = pd.merge(accounts, usage_summary, on='account_id', how='left')
df = pd.merge(df, ticket_summary, on='account_id', how='left')

# 5. [보강] 파생 변수 생성 (Feature Engineering)
# 가입 기간 (현재 날짜를 2025-01-01이라 가정하거나 데이터 내 최대 날짜 기준)
ref_date = df['signup_date'].max()
df['tenure_days'] = (ref_date - df['signup_date']).dt.days

# 6. [보강] 결측치 및 플래그 처리
# 이용 경험이 없는 경우
df['is_inactive_user'] = df['total_usage_cnt'].isna().astype(int)
for col in ['total_usage_cnt', 'total_duration', 'total_errors', 'feature_diversity', 'used_beta', 'total_tickets']:
    df[col] = df[col].fillna(0)

# 만족도 결측 플래그 및 대체
df['has_satisfaction_score'] = df['avg_satisfaction'].notna().astype(int)
df['avg_satisfaction'] = df['avg_satisfaction'].fillna(-1)
df['avg_resolution_time'] = df['avg_resolution_time'].fillna(-1)

# 7. 범주형 데이터 처리 및 불필요 컬럼 제거
# 데이터 누수 방지 및 식별자 제거
drop_cols = ['account_id', 'account_name', 'signup_date', 'country']
df_model = df.drop(columns=drop_cols)

# 원-핫 인코딩
df_model = pd.get_dummies(df_model, columns=['industry', 'plan_tier', 'referral_source'], drop_first=True, dtype=int)

# 8. 최종 결과 저장
print("최종 생성된 특성(Feature) 개수:", len(df_model.columns) - 1)
df_model.to_csv('01_preprocessed_data.csv', index=False)
print("전처리 완료 및 저장 성공.")

최종 생성된 특성(Feature) 개수: 23
전처리 완료 및 저장 성공.
